Makemore NLP Language Model

This project is similar to the previous in the sense that its using the same data set and has the same task of predicting the next character. However, instead of only taking the previous character as context for predicting the next, we will increase the context length to N (typically N=3). Additionally, the architecture of the model will be slightly more complicated, instead of a single matmul we will design a Multi-Layer Perceptron Model


Step 1: Imports, read names.txt, and build chars,stoi,itos (as previously done)

In [76]:
import torch, torch.nn.functional as F, matplotlib.pyplot as plt

words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

print(len(words), words[:5])
print(stoi)
print(itos[5])
print(chars[:5])

32033 ['emma', 'olivia', 'ava', 'isabella', 'sophia']
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
e
['a', 'b', 'c', 'd', 'e']


Previously the dataset was a list of pairs (prev_char, next_char). Now its a list of pairs (Prev_N_chars, next_char), where N= block_size. Lets start .emma. as an example. (..., e) -> (..e,m) -> (.em, m) -> (emm,a) -> (mma, .). Notice, since the word length is 4, for a couple of the examples we needed to pad with '.'. Now lets create this dataset

In [77]:
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

print(X.shape, X.dtype)
print(Y.shape, Y.dtype)
print(X[:10])
print(Y[:10])

for x,y in zip(X[:20], Y[:20]):
     print(''.join(itos[i.item()] for i in x), '-->', itos[y.item()])


torch.Size([228146, 3]) torch.int64
torch.Size([228146]) torch.int64
tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22]])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9])
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
... --> a
..a --> v
.av --> a
ava --> .
... --> i
..i --> s
.is --> a
isa --> b


Step 2: Create the lookup table (embedding table).

We want an embedding for each of the 27 characters. Therefore, we will have 27 rows. For now, lets assume each embedding is 2-Dimensional. Therefore, the lookup table will have shape (27,2). Initially, these will be random numbers

In [78]:
C=torch.randn((27,2), requires_grad=True)
emb = C[X]

print(C.shape)
print(emb.shape)
print(emb[0])
print(emb[0,0])
print(C[X[0,0]])

torch.Size([27, 2])
torch.Size([228146, 3, 2])
tensor([[ 0.4209, -0.5452],
        [ 0.4209, -0.5452],
        [ 0.4209, -0.5452]], grad_fn=<SelectBackward0>)
tensor([ 0.4209, -0.5452], grad_fn=<SelectBackward0>)
tensor([ 0.4209, -0.5452], grad_fn=<SelectBackward0>)


The lookup table has a 2-Dimensional embedding for each of the characters. Therefore the embedding for one character has the shape (2,). If we take a single data point which has 3 characters, each character has its own vector embedding hence the shape of the vector embedding for one data point becomes (3,2). Therefore for the whole dataset of N examples, all these examples are stacked to create a shape (N,3,2)

Step 3: Flatten + Hidden Layer

After the lookup table, the next layer of the MLP is the hidden layer (w1x1 + w2x2 + w3x3.... wNxN + bias). However, the issue that we have now is that our input is 3-Dimensional (N,3,2). Which means, before applying the weights (matmul), we need to flatten it into a 2-Dimensional input

First we start with flattening, the 3-Dimensional input. Take it from (N,3,2) -> (N,6)

In [79]:
x=emb.view(-1,6)

Then we build the hidden layer of the MLP. Recall that we first multiply by a set of weights, then we add the biases, then we run it through a nonlinearity (lets assume its tanh).

In [80]:
n_hidden = 100
W1=torch.randn(6,n_hidden, requires_grad=True)
b1=torch.randn(n_hidden, requires_grad=True)

h=torch.tanh(x @ W1 + b1)

h.shape

torch.Size([228146, 100])

Now, the output layer. Previously, we took the initial 6 features up to 100 through the 100 neurons. Now, we want to take the features down to 27, hence our second layer of neurons will have 27 neurons.

In [81]:
n_hidden2 = 27
W2=torch.randn(n_hidden,n_hidden2, requires_grad=True)
b2=torch.randn(n_hidden2, requires_grad=True)

logits = h @ W2 + b2

logits.shape

torch.Size([228146, 27])

Now, the 27 outputs that we have are raw, unbounded scores (What we call logits), theyre not probabilities yet. We want to turn them into probabilities. We can do this by taking each output and first turning them all positive, we can do this through the exponential function, then we divide it by the sum of all the outputs (We did something similar in Bigram model). P = N.float() / N.sum(1, keepdim=True)


In [82]:
counts = torch.exp(logits)
P = counts / counts.sum(1, keepdim=True)


In [83]:
correct_probs = P[torch.arange(Y.shape[0]),Y]

neg_log = -torch.log(correct_probs)
mean_neg_log = torch.mean(neg_log)

print(mean_neg_log)


tensor(16.9615, grad_fn=<MeanBackward0>)


Now, we have a full working forward pass with an initial loss of 17.4117. Now, we need to run a backward pass and iteratively change the weights to reduce this overall loss

lets create a whole learning loop

In [84]:
num_iterations = 100
learning_rate = 1

for k in range(num_iterations):
    #Forward Pass
    emb = C[X]
    x = emb.view(-1,6)
    h=torch.tanh(x @ W1 + b1)
    logits = h @ W2 + b2
    counts = torch.exp(logits)
    P = counts / counts.sum(1, keepdim=True)

    #Loss
    correct_probs = P[torch.arange(Y.shape[0]),Y]
    neg_log = -torch.log(correct_probs)
    mean_neg_log = torch.mean(neg_log)

    #Backward Pass
    mean_neg_log.backward()


    #Update
    C.data += learning_rate * -C.grad
    W1.data += learning_rate * -W1.grad
    W2.data += learning_rate * -W2.grad
    b1.data += learning_rate * -b1.grad
    b2.data += learning_rate * -b2.grad
    C.grad.zero_()
    W1.grad.zero_()
    W2.grad.zero_()
    b1.grad.zero_()
    b2.grad.zero_()

    #Logging
    print((k,mean_neg_log.item()))


(0, 16.961532592773438)
(1, 12.908673286437988)
(2, 11.81521987915039)
(3, 11.023468971252441)
(4, 8.447492599487305)
(5, 8.167224884033203)
(6, 7.865389347076416)
(7, 11.070924758911133)
(8, 8.476517677307129)
(9, 8.161782264709473)
(10, 8.624892234802246)
(11, 8.132495880126953)
(12, 7.792340278625488)
(13, 8.543338775634766)
(14, 7.142391681671143)
(15, 7.581628799438477)
(16, 7.033602237701416)
(17, 6.899615287780762)
(18, 6.656770706176758)
(19, 6.746387004852295)
(20, 6.548229694366455)
(21, 5.837528228759766)
(22, 5.892368316650391)
(23, 5.817343711853027)
(24, 7.006779670715332)
(25, 6.589452743530273)
(26, 6.153995513916016)
(27, 5.094214916229248)
(28, 6.801288604736328)
(29, 5.693264007568359)
(30, 5.011837482452393)
(31, 5.3470778465271)
(32, 5.875244140625)
(33, 8.486891746520996)
(34, 8.343330383300781)
(35, 5.924693584442139)
(36, 6.096419811248779)
(37, 5.456202030181885)
(38, 5.166162014007568)
(39, 4.670877456665039)
(40, 4.398838520050049)
(41, 4.772592067718506)
(42

Next: 1)Replace the manual loss calculation with the cross_entropy() function. 2)Introduce mini-batches when training